# 🔧 Notebook 02: Preprocessing & Augmentation
## Chest Cancer Classification using CNN and MLOps

**Goals of this notebook:**
1. Resize images to (224, 224)
2. Normalize pixel values to [0, 1]
3. Apply data augmentation (rotation, zoom, flip)
4. Build ImageDataGenerators for train/val/test
5. Visualize augmented samples


In [ ]:
# ─── Imports ─────────────────────────────────────────────────────
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, load_img, img_to_array
)

sys.path.append('..')

plt.style.use('seaborn-v0_8-whitegrid')

print(f'TensorFlow version: {tf.__version__}')
print('✅ Imports successful!')

In [ ]:
# ─── Config ──────────────────────────────────────────────────────
IMAGE_SIZE  = (224, 224)   # VGG16 / ResNet50 compatible
BATCH_SIZE  = 32
RAW_DIR     = Path('../data/raw')
SEED        = 42

TRAIN_DIR = RAW_DIR / 'train'
VAL_DIR   = RAW_DIR / 'valid'
TEST_DIR  = RAW_DIR / 'test'

print(f'Image size:  {IMAGE_SIZE}')
print(f'Batch size:  {BATCH_SIZE}')
print(f'Train path:  {TRAIN_DIR}')
print(f'Val path:    {VAL_DIR}')
print(f'Test path:   {TEST_DIR}')

## 🔄 Step 1: Build Augmented Training Generator

Data augmentation creates artificial variations of training images to:
- Prevent overfitting
- Simulate real-world image variation
- Improve model generalization on unseen CT scans

In [ ]:
# ─── Training generator WITH augmentation ────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,          # Normalize: [0,255] → [0,1]
    rotation_range=20,          # Random rotation ±20°
    zoom_range=0.2,             # Random zoom 80%-120%
    horizontal_flip=True,       # Mirror images horizontally
    width_shift_range=0.1,      # Horizontal shift ±10%
    height_shift_range=0.1,     # Vertical shift ±10%
    shear_range=0.1,            # Shear transformation
    brightness_range=[0.8, 1.2],# Random brightness
    fill_mode='nearest'         # Fill empty pixels
)

# ─── Val/Test generator WITHOUT augmentation ─────────────────────
val_datagen  = ImageDataGenerator(rescale=1.0 / 255)   # Only normalize
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

print('✅ Data generators configured!')
print('\nAugmentation parameters:')
print('  • Rotation:          ±20°')
print('  • Zoom:              80%–120%')
print('  • Horizontal flip:   Yes')
print('  • Width/Height shift: ±10%')
print('  • Brightness:        80%–120%')
print('  • Normalization:     [0, 1]')

In [ ]:
# ─── Flow from directory ─────────────────────────────────────────
if TRAIN_DIR.exists():
    train_generator = train_datagen.flow_from_directory(
        str(TRAIN_DIR),
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',  # One-hot labels
        shuffle=True,
        seed=SEED
    )

    val_generator = val_datagen.flow_from_directory(
        str(VAL_DIR),
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    test_generator = test_datagen.flow_from_directory(
        str(TEST_DIR),
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    print(f'\n✅ Generators ready!')
    print(f'  Classes: {train_generator.class_indices}')
    print(f'  Train batches:      {len(train_generator)}')
    print(f'  Validation batches: {len(val_generator)}')
    print(f'  Test batches:       {len(test_generator)}')
else:
    print('⚠️  Dataset not found — creating demo generator for illustration')
    train_generator = None

## 🖼️ Step 2: Visualize Augmented Images

In [ ]:
# ─── Show augmented versions of one image ────────────────────────
def show_augmented_images(image_path: str, n_augmented: int = 8):
    """
    Load one image and show multiple augmented versions.
    Demonstrates what the model 'sees' during training.
    """
    img     = load_img(image_path, target_size=IMAGE_SIZE)
    img_arr = img_to_array(img)
    img_arr = np.expand_dims(img_arr, axis=0)  # Add batch dimension
    
    aug_gen = ImageDataGenerator(
        rescale=1.0/255,
        rotation_range=30,
        zoom_range=0.3,
        horizontal_flip=True,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        brightness_range=[0.6, 1.4],
        fill_mode='nearest'
    )
    
    n_cols = 4
    n_rows = (n_augmented + 1 + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    axes = axes.flatten()
    
    # Show original
    axes[0].imshow(img)
    axes[0].set_title('Original', fontweight='bold', color='red')
    axes[0].axis('off')
    
    # Show augmented
    gen = aug_gen.flow(img_arr, batch_size=1)
    for i in range(1, n_augmented + 1):
        aug_img = next(gen)[0]
        axes[i].imshow(aug_img)
        axes[i].set_title(f'Augmented #{i}', fontsize=9)
        axes[i].axis('off')
    
    for j in range(n_augmented + 1, len(axes)):
        axes[j].axis('off')
    
    plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Find a real image to demonstrate augmentation
sample_image = None
for split in ['train', 'valid', 'test']:
    for cls_dir in (RAW_DIR / split).glob('*/'):
        imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
        if imgs:
            sample_image = str(imgs[0])
            break
    if sample_image:
        break

if sample_image:
    print(f'Using sample image: {sample_image}')
    show_augmented_images(sample_image, n_augmented=8)
else:
    print('⚠️  No images found. Please add dataset to data/raw/')

## ⚖️ Step 3: Handle Class Imbalance

In [ ]:
# ─── Compute class weights ───────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

if train_generator is not None:
    y_labels = train_generator.classes
    classes  = np.unique(y_labels)
    
    weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=y_labels
    )
    
    class_weight_dict = dict(zip(classes, weights))
    idx_to_class = {v: k for k, v in train_generator.class_indices.items()}
    
    print('⚖️  Class Weights (for handling imbalance):')
    print('=' * 40)
    for idx, w in class_weight_dict.items():
        print(f'  {idx_to_class[idx]:<35} → {w:.4f}')
    
    print('\n💡 Tip: Higher weight = more focus on under-represented class')
else:
    class_weight_dict = {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0}
    print('Using equal class weights (no dataset found)')

## 🔍 Step 4: Pixel Value Analysis

In [ ]:
# ─── Show pixel distribution before/after normalization ──────────
if sample_image:
    img_raw  = load_img(sample_image, target_size=IMAGE_SIZE)
    arr_raw  = img_to_array(img_raw)          # [0, 255]
    arr_norm = arr_raw / 255.0                # [0, 1]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Original image
    axes[0].imshow(img_raw.resize((224, 224)) if hasattr(img_raw, 'resize') else img_raw)
    axes[0].set_title('Original Image', fontweight='bold')
    axes[0].axis('off')
    
    # Before normalization
    axes[1].hist(arr_raw.ravel(), bins=50, color='tomato', alpha=0.8, edgecolor='black')
    axes[1].set_title(f'Before Normalization\nRange: [{arr_raw.min():.0f}, {arr_raw.max():.0f}]',
                      fontweight='bold')
    axes[1].set_xlabel('Pixel Value')
    axes[1].set_ylabel('Frequency')
    
    # After normalization
    axes[2].hist(arr_norm.ravel(), bins=50, color='steelblue', alpha=0.8, edgecolor='black')
    axes[2].set_title(f'After Normalization\nRange: [{arr_norm.min():.2f}, {arr_norm.max():.2f}]',
                      fontweight='bold')
    axes[2].set_xlabel('Pixel Value (normalized)')
    axes[2].set_ylabel('Frequency')
    
    plt.suptitle('Pixel Value Distribution: Before vs After Normalization',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
    print('✅ Normalization maps [0,255] → [0,1] for better gradient flow in CNN')
else:
    print('⚠️  No sample image to analyze.')

## ✅ Summary

| Step | Action | Result |
|------|--------|--------|
| Resize | 224×224 pixels | Uniform input for CNN |
| Normalize | ÷ 255 | [0, 1] range |
| Augmentation | Rotation, Zoom, Flip | Bigger virtual dataset |
| Class Weights | Balanced weighting | Fair learning across classes |

➡️ **Next:** Open `03_cnn_training.ipynb`